# Đồ án cuối kỳ — Image Captioning Flickr8k (train trên Colab)

Notebook mỏng: chỉ gọi các CLI trong package `final/` (toàn bộ logic nằm trong repo, đã test 44/44).

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**.

Artifact (features, checkpoints, outputs) được ghi thẳng vào Google Drive qua symlink —
rớt session chạy lại từ đầu notebook là tiếp tục được (mọi bước đều idempotent,
train chỉ bị bỏ qua khi có ĐỦ checkpoint + history.json, tức là run đã hoàn thành thật).

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 2. Mount Google Drive (lưu artifact bền vững)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/final_captioning/{data,checkpoints,outputs}

In [ ]:
# 3. Lấy code (repo public thì giữ nguyên; repo private thì thay bằng
#    https://<TOKEN>@github.com/tuan1kdt/deep-learning.git)
import os
if not os.path.exists('/content/deepLearning'):
    !git clone https://github.com/tuan1kdt/deep-learning.git /content/deepLearning
%cd /content/deepLearning
!git pull

In [ ]:
# 4. Cài dependency (torch/torchvision/matplotlib Colab có sẵn)
!pip -q install datasets nltk pycocoevalcap

In [ ]:
# 5. Trỏ artifact về Drive bằng symlink — mọi thứ nặng đều sống sót qua rớt session.
#    (final/data chứa dataset ~1.1GB + features ~1.5GB; tải/precompute lại chỉ ~15 phút
#     nếu bạn muốn để trong VM cho nhanh I/O thì bỏ symlink data, chỉ giữ checkpoints+outputs.)
import os
for name in ['data', 'checkpoints', 'outputs']:
    local = f'/content/deepLearning/final/{name}'
    target = f'/content/drive/MyDrive/final_captioning/{name}'
    if not os.path.islink(local):
        if os.path.isdir(local):
            !rm -rf {local}
        os.symlink(target, local)
    print(name, '->', os.path.realpath(local))

In [ ]:
# 6. Chạy TOÀN BỘ pipeline: download → vocab → features → 3 run train →
#    evaluate (greedy/beam3/beam5 × 3 checkpoint) → visualize.
#    PY=python vì Colab không dùng .venv. Chạy lại cell này sau khi rớt session là resume.
!PY=python bash final/run_all.sh

In [ ]:
# 7. (Stretch — chạy sau khi bước 6 xong) SCST fine-tune từ checkpoint tốt nhất
#    theo CIDEr trên bảng eval của bước 6, rồi evaluate lại.
!PY=python python -m final.scst --checkpoint final/checkpoints/lstm.pt --epochs 3
!python -m final.evaluate --checkpoint final/checkpoints/lstm_scst.pt

## Lấy kết quả về máy

Mọi thứ đã nằm trong Drive `final_captioning/{checkpoints,outputs}`.
Về Mac: tải 2 thư mục đó và đặt vào `final/checkpoints/`, `final/outputs/`
(hoặc dùng cell dưới nén lại tải một file duy nhất).

In [ ]:
# 8. Nén kết quả (không kèm features nặng) để tải về
!cd /content/drive/MyDrive/final_captioning && zip -r results.zip checkpoints outputs -x '*.pt.tmp'
print('=> Drive/final_captioning/results.zip')